# Pendulum Control with Model Predictive Control and Neural Network Imitation Learning

**Portfolio version for GitHub**

This project compares three control strategies for the `Pendulum-v1` reinforcement-learning environment:

1. **Random controller** as a baseline.
2. **Model Predictive Control (MPC)** using evolutionary optimization.
3. **Neural Network controller** trained to imitate MPC actions.

The notebook is structured like a data science project: problem framing, reproducible setup, data generation, exploratory analysis, model training, evaluation, and conclusions.

## 1. Project overview

The objective is to swing and stabilize a pendulum near the upright position while minimizing control effort. Instead of only showing a working controller, this notebook treats the assignment as an end-to-end analytics workflow:

- Simulate control trajectories.
- Generate a supervised learning dataset from MPC behavior.
- Analyze state/action distributions.
- Train a neural network policy.
- Compare policies using repeated experiments.
- Report performance using summary statistics and visualizations.

**Key technical topics:** simulation, optimization, supervised learning, imitation learning, model evaluation, experiment design.

In [ ]:
# Optional installs for fresh environments such as Google Colab.
# In a local GitHub project, keep these packages in requirements.txt instead.

# %pip install -q gymnasium scipy skorch torch scikit-learn pandas matplotlib

In [ ]:
import random
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
import torch
from torch import nn

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skorch import NeuralNetRegressor

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ENV_NAME = "Pendulum-v1"
G = 9.81
MAX_TORQUE = 2.0
EPISODE_STEPS = 200

def make_env(render_mode=None):
    """Create a reproducible Pendulum environment."""
    env = gym.make(ENV_NAME, g=G, render_mode=render_mode)
    env.reset(seed=SEED)
    env.action_space.seed(SEED)
    env.observation_space.seed(SEED)
    return env

env = make_env()
env

## 2. Pendulum environment

The observation vector is:

- `cos(theta)`
- `sin(theta)`
- angular velocity

The action is torque, clipped by the environment to approximately `[-2, 2]`.

The reward is higher when the pendulum is close to the upright position, moves slowly, and uses less torque. Since the environment returns reward and the MPC optimizer minimizes a cost, this project defines a cost function that mirrors the reward objective.

In [ ]:
@dataclass
class SimulationResult:
    score: float
    states: np.ndarray
    actions: np.ndarray
    rewards: np.ndarray


def angle_from_obs(obs: np.ndarray) -> float:
    """Convert Pendulum observation [cos(theta), sin(theta), theta_dot] to angle in radians."""
    return float(np.arctan2(obs[1], obs[0]))


def pendulum_cost(obs: np.ndarray, action: float) -> float:
    """Cost equivalent to the negative Pendulum reward structure."""
    theta = angle_from_obs(obs)
    theta_dot = float(obs[2])
    torque = float(np.clip(action, -MAX_TORQUE, MAX_TORQUE))
    return theta**2 + 0.1 * theta_dot**2 + 0.001 * torque**2


def set_initial_state(env, angle_rad: float, angular_velocity: float) -> np.ndarray:
    """Reset the environment and set a custom physical state."""
    obs, _ = env.reset()
    env.unwrapped.state = np.array([angle_rad, angular_velocity], dtype=np.float64)
    obs = np.array([np.cos(angle_rad), np.sin(angle_rad), angular_velocity], dtype=np.float64)
    return obs

## 3. Controllers

This section defines three policies:

- **Random:** samples torque uniformly.
- **MPC:** searches for the best sequence of future torques, then applies only the first action.
- **Neural network:** predicts torque directly from the current state.

A key improvement over the assignment version is that the controller logic is now explicit and mutually exclusive, which avoids accidental fall-through between controller types.

In [ ]:
def random_controller(obs: np.ndarray) -> float:
    """Baseline policy: choose a random torque."""
    return float(np.random.uniform(-MAX_TORQUE, MAX_TORQUE))


class MPCProblem:
    """Finite-horizon cost function for pendulum Model Predictive Control.

    The original assignment used PyGMO. For a public GitHub project, SciPy is a
    better default because it installs reliably across Windows, macOS, Linux,
    Google Colab, and GitHub Codespaces.
    """

    def __init__(self, initial_state: np.ndarray, horizon: int = 10):
        self.initial_state = np.array(initial_state, dtype=np.float64)
        self.horizon = horizon

    def __call__(self, torques: np.ndarray) -> float:
        """Return cumulative cost for a candidate sequence of torques."""
        env_local = make_env()
        theta = angle_from_obs(self.initial_state)
        theta_dot = float(self.initial_state[2])
        set_initial_state(env_local, theta, theta_dot)

        total_cost = 0.0

        for torque in torques:
            action = np.array([np.clip(torque, -MAX_TORQUE, MAX_TORQUE)], dtype=np.float32)
            _, reward, terminated, truncated, _ = env_local.step(action)
            total_cost += -float(reward)

            if terminated or truncated:
                break

        env_local.close()
        return total_cost

    @property
    def bounds(self) -> List[Tuple[float, float]]:
        return [(-MAX_TORQUE, MAX_TORQUE)] * self.horizon


def mpc_controller(
    obs: np.ndarray,
    horizon: int = 10,
    max_iterations: int = 20,
    population_size: int = 10,
) -> float:
    """Return the first action from an optimized torque sequence.

    Differential evolution is a derivative-free optimizer, which is suitable here
    because the simulator is treated as a black-box function.
    """
    objective = MPCProblem(obs, horizon=horizon)

    result = differential_evolution(
        objective,
        bounds=objective.bounds,
        maxiter=max_iterations,
        popsize=population_size,
        seed=SEED,
        polish=False,
        updating="immediate",
        workers=1,
    )

    best_sequence = result.x
    return float(np.clip(best_sequence[0], -MAX_TORQUE, MAX_TORQUE))


In [ ]:
def play_episode(
    controller: Callable[[np.ndarray], float],
    initial_angle: float,
    initial_velocity: float,
    max_steps: int = EPISODE_STEPS,
) -> SimulationResult:
    """Run one episode and collect states, actions, and rewards."""
    env_local = make_env()
    obs = set_initial_state(env_local, initial_angle, initial_velocity)

    states, actions, rewards = [], [], []
    total_reward = 0.0

    for _ in range(max_steps):
        torque = float(controller(obs))
        action = np.array([np.clip(torque, -MAX_TORQUE, MAX_TORQUE)], dtype=np.float32)

        states.append(obs.copy())
        actions.append(float(action[0]))

        obs, reward, terminated, truncated, _ = env_local.step(action)
        rewards.append(float(reward))
        total_reward += float(reward)

        if terminated or truncated:
            break

    env_local.close()

    return SimulationResult(
        score=total_reward,
        states=np.array(states),
        actions=np.array(actions),
        rewards=np.array(rewards),
    )


def sample_initial_condition() -> Tuple[float, float]:
    """Sample initial states close to the upright region used in the original assignment."""
    angle = np.random.uniform(np.deg2rad(-20), np.deg2rad(20))
    velocity = np.random.uniform(-0.1, 0.1)
    return angle, velocity


def evaluate_controller(controller, n_episodes: int = 50) -> pd.DataFrame:
    """Evaluate a controller over repeated random initial conditions."""
    records = []

    for episode in range(n_episodes):
        angle, velocity = sample_initial_condition()
        result = play_episode(controller, angle, velocity)

        records.append({
            "episode": episode,
            "initial_angle_deg": np.rad2deg(angle),
            "initial_velocity": velocity,
            "score": result.score,
            "mean_abs_torque": np.mean(np.abs(result.actions)),
            "final_angle_deg": np.rad2deg(angle_from_obs(result.states[-1])),
            "steps": len(result.rewards),
        })

    return pd.DataFrame(records)

## 4. Baseline experiment

A baseline is necessary because a portfolio project should show whether the proposed model actually improves anything. Here, the random controller provides the minimum benchmark.

In [ ]:
random_scores = evaluate_controller(random_controller, n_episodes=30)
random_scores.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(random_scores["score"], bins=12)
ax.set_title("Random controller score distribution")
ax.set_xlabel("Episode score")
ax.set_ylabel("Frequency")
plt.show()

## 5. Generate training data from MPC

The neural network controller is trained using **imitation learning**. MPC acts as the expert policy, and the neural network learns the mapping:

`state -> torque`

To improve dataset quality, this workflow keeps only trajectories above a configurable score threshold.

In [ ]:
def generate_mpc_dataset(
    n_episodes: int = 50,
    min_score: float = -250,
    horizon: int = 10,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate supervised learning data from MPC trajectories."""
    rows = []
    episode_rows = []

    mpc_policy = lambda obs: mpc_controller(obs, horizon=horizon)

    for episode in range(n_episodes):
        angle, velocity = sample_initial_condition()
        result = play_episode(mpc_policy, angle, velocity)

        episode_rows.append({
            "episode": episode,
            "score": result.score,
            "n_steps": len(result.actions),
            "accepted": result.score >= min_score,
        })

        if result.score < min_score:
            continue

        for step, (state, action, reward) in enumerate(zip(result.states, result.actions, result.rewards)):
            rows.append({
                "episode": episode,
                "step": step,
                "cos_theta": state[0],
                "sin_theta": state[1],
                "theta_dot": state[2],
                "angle_rad": angle_from_obs(state),
                "angle_deg": np.rad2deg(angle_from_obs(state)),
                "torque": action,
                "reward": reward,
            })

    return pd.DataFrame(rows), pd.DataFrame(episode_rows)


# This may take several minutes depending on your machine.
# For a quick demo, reduce n_episodes to 10.
mpc_data, mpc_episode_summary = generate_mpc_dataset(n_episodes=50, min_score=-250, horizon=10)

print(f"Accepted training samples: {len(mpc_data):,}")
print(f"Accepted episodes: {mpc_episode_summary['accepted'].sum()} / {len(mpc_episode_summary)}")
mpc_episode_summary.head()

## 6. Exploratory data analysis

Before training, inspect the expert demonstrations. This makes the notebook more useful as a data analytics project, not only a simulation assignment.

In [ ]:
mpc_data[["cos_theta", "sin_theta", "theta_dot", "angle_deg", "torque", "reward"]].describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mpc_data["torque"], bins=30)
ax.set_title("Distribution of MPC expert actions")
ax.set_xlabel("Torque")
ax.set_ylabel("Frequency")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(mpc_data["angle_deg"], mpc_data["torque"], alpha=0.25, s=10)
ax.set_title("Relationship between pendulum angle and MPC torque")
ax.set_xlabel("Angle [degrees]")
ax.set_ylabel("Torque")
plt.show()

In [ ]:
corr = mpc_data[["cos_theta", "sin_theta", "theta_dot", "angle_deg", "torque", "reward"]].corr(numeric_only=True)
corr.style.background_gradient(cmap="coolwarm").format("{:.2f}")

## 7. Neural network policy model

The neural network predicts the expert MPC torque from the current state. This is a supervised regression problem.

Portfolio improvements added here:

- Train/validation/test split.
- Standardized features.
- Reproducible neural network training.
- Regression metrics.
- Hyperparameter search option.

In [ ]:
class TorquePolicyNN(nn.Module):
    """Feedforward network for torque prediction."""

    def __init__(self, hidden_units: int = 64, dropout: float = 0.0):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        # Tanh returns [-1, 1], scaled to the valid torque range [-2, 2].
        return MAX_TORQUE * self.network(x)

In [ ]:
features = ["cos_theta", "sin_theta", "theta_dot"]
target = "torque"

X = mpc_data[features].astype("float32").values
y = mpc_data[[target]].astype("float32").values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

net = NeuralNetRegressor(
    module=TorquePolicyNN,
    module__hidden_units=64,
    module__dropout=0.0,
    max_epochs=30,
    lr=0.001,
    batch_size=128,
    optimizer=torch.optim.Adam,
    criterion=nn.MSELoss,
    iterator_train__shuffle=True,
    train_split=None,
    verbose=0,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

policy_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", net),
])

policy_pipeline.fit(X_train, y_train)

In [ ]:
def regression_report(model, X_eval, y_eval, split_name: str) -> Dict[str, float]:
    preds = model.predict(X_eval)
    return {
        "split": split_name,
        "MAE": mean_absolute_error(y_eval, preds),
        "RMSE": mean_squared_error(y_eval, preds, squared=False),
        "R2": r2_score(y_eval, preds),
    }

metrics = pd.DataFrame([
    regression_report(policy_pipeline, X_train, y_train, "train"),
    regression_report(policy_pipeline, X_val, y_val, "validation"),
    regression_report(policy_pipeline, X_test, y_test, "test"),
])

metrics

In [ ]:
history = policy_pipeline["model"].history
train_loss = history[:, "train_loss"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(train_loss) + 1), train_loss)
ax.set_title("Neural network training loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
plt.show()

In [ ]:
test_predictions = policy_pipeline.predict(X_test).reshape(-1)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test.reshape(-1), test_predictions, alpha=0.35, s=12)
ax.plot([-MAX_TORQUE, MAX_TORQUE], [-MAX_TORQUE, MAX_TORQUE])
ax.set_title("Predicted vs actual MPC torque")
ax.set_xlabel("Actual torque")
ax.set_ylabel("Predicted torque")
plt.show()

### Optional: hyperparameter search

Run this cell when you want a stronger final model. It is separated because grid search can be slow.

In [ ]:
# param_grid = {
#     "model__module__hidden_units": [32, 64, 128],
#     "model__module__dropout": [0.0, 0.1],
#     "model__lr": [0.001, 0.003],
#     "model__batch_size": [64, 128],
# }
#
# grid_search = GridSearchCV(
#     estimator=policy_pipeline,
#     param_grid=param_grid,
#     scoring="neg_mean_squared_error",
#     cv=3,
#     verbose=2,
# )
#
# grid_search.fit(X_train, y_train)
# print(grid_search.best_params_)
# policy_pipeline = grid_search.best_estimator_

## 8. Evaluate the learned controller in simulation

The supervised metrics show whether the model copies MPC actions well. The real test is whether it controls the pendulum successfully in the environment.

In [ ]:
def nn_controller(obs: np.ndarray) -> float:
    obs_2d = np.array(obs, dtype=np.float32).reshape(1, -1)
    torque = policy_pipeline.predict(obs_2d)[0, 0]
    return float(np.clip(torque, -MAX_TORQUE, MAX_TORQUE))


nn_scores = evaluate_controller(nn_controller, n_episodes=100)
random_scores = evaluate_controller(random_controller, n_episodes=100)

comparison = pd.concat([
    random_scores.assign(controller="Random"),
    nn_scores.assign(controller="Neural Network"),
], ignore_index=True)

comparison.groupby("controller").agg(
    mean_score=("score", "mean"),
    median_score=("score", "median"),
    std_score=("score", "std"),
    mean_abs_torque=("mean_abs_torque", "mean"),
    mean_final_angle_deg=("final_angle_deg", "mean"),
).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for controller_name, group in comparison.groupby("controller"):
    ax.hist(group["score"], bins=20, alpha=0.55, label=controller_name)

ax.set_title("Controller performance comparison")
ax.set_xlabel("Episode score")
ax.set_ylabel("Frequency")
ax.legend()
plt.show()

In [ ]:
example_angle, example_velocity = sample_initial_condition()
example_random = play_episode(random_controller, example_angle, example_velocity)
example_nn = play_episode(nn_controller, example_angle, example_velocity)

def trajectory_frame(result: SimulationResult, controller_name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "step": np.arange(len(result.states)),
        "angle_deg": [np.rad2deg(angle_from_obs(s)) for s in result.states],
        "theta_dot": result.states[:, 2],
        "torque": result.actions,
        "controller": controller_name,
    })

trajectory = pd.concat([
    trajectory_frame(example_random, "Random"),
    trajectory_frame(example_nn, "Neural Network"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 4))
for controller_name, group in trajectory.groupby("controller"):
    ax.plot(group["step"], group["angle_deg"], label=controller_name)

ax.axhline(0, linestyle=":")
ax.set_title("Example trajectory: pendulum angle over time")
ax.set_xlabel("Step")
ax.set_ylabel("Angle [degrees]")
ax.legend()
plt.show()

## 9. Conclusions

This project demonstrates how an optimization-based controller can be used to generate expert demonstrations, and how a neural network can learn a faster approximate policy from those demonstrations.

Main takeaways:

- The random controller provides a weak baseline and produces unstable behavior.
- MPC generates higher-quality control demonstrations but is computationally expensive.
- The neural network imitates MPC and can be evaluated both as a regression model and as a controller in simulation.
- The strongest portfolio extension would be to compare NN, MPC, and random controllers over larger evaluation sets and report confidence intervals.

## 10. Suggested GitHub repository structure

```text
pendulum-control-data-science/
├── notebooks/
│   └── pendulum_mpc_nn_portfolio.ipynb
├── src/
│   ├── controllers.py
│   ├── simulation.py
│   └── models.py
├── data/
│   └── .gitkeep
├── requirements.txt
├── README.md
└── .gitignore
```

The notebook is self-contained, but splitting functions into `src/` files would make the project cleaner and more professional.

## Future work

Potential improvements:

- Add confidence intervals for controller scores.
- Save trained models with `joblib` or `torch.save`.
- Add experiment tracking using MLflow or Weights & Biases.
- Try reinforcement learning algorithms such as DDPG, PPO, or SAC.
- Package the controllers into reusable Python modules.
- Add automated tests for the cost function, controller output bounds, and simulation loop.